# Cell tracking — 検出位置と正解の時間変化

exp015 の train 全199件の検出候補と、同名の GEFF / Zarr を比較する EDA ビューアーです。
**Edit を開いて Run All** すると、時刻・Z断面・正解の系譜を操作できます。
保存された静的な Notebook ページでは Python の操作部は動きません。

- CPU / Internet OFF で実行。画像は選択時刻だけ読み、ローカルへダウンロードしません。
- Input: `biohub-cell-tracking-during-development` と
  `kentookumura/exp015-oracle-stage-limits-inference` の出力。
- 正解は疎です。未対応の検出は誤検出とは限りません。
- 候補 ID は各検出点の識別子です。同一細胞を時間方向につなぐ ID ではありません。
- 公開モデルは train 199件を学習に含むため、この表示は固定公開モデル下の診断です。
- 対応表の保存ボタンは作業ディレクトリに CSV と設定 JSON を作ります。


In [ ]:
from pathlib import Path
import sys
import json

BASE = Path('/kaggle/working') if Path('/kaggle/input').exists() else Path.cwd()
PACKAGE = BASE / 'tracking_viewer_app'
(PACKAGE / 'app').mkdir(parents=True, exist_ok=True)
(PACKAGE / 'app' / '__init__.py').write_text('')
FILES = {'tracking_data.py': '"""Read-only inputs and spatial diagnostics for the tracking viewer.\n\nDistances and plots use micrometers; GEFF stores original voxel coordinates.\nNo model inference, prediction edges, or official competition score is computed.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import linear_sum_assignment\nfrom scipy.spatial.distance import cdist\n\nXYZ = ["z", "y", "x"]\nNODE_COLUMNS = ["node_id", "t", *XYZ]\n\n\ndef validate_nodes(nodes: pd.DataFrame) -> pd.DataFrame:\n    nodes = nodes.copy()\n    if not set(NODE_COLUMNS).issubset(nodes):\n        raise ValueError(f"Nodes require {NODE_COLUMNS}")\n    values = nodes[NODE_COLUMNS].to_numpy(dtype=float)\n    if not np.isfinite(values).all():\n        raise ValueError("Node IDs, frames and coordinates must be finite")\n    if nodes.node_id.duplicated().any():\n        raise ValueError("Node IDs must be unique within a dataset")\n    for col in ["node_id", "t"]:\n        if (nodes[col] < 0).any() or (nodes[col] % 1 != 0).any():\n            raise ValueError(f"{col} must contain nonnegative integers")\n        nodes[col] = nodes[col].astype(np.int64)\n    return nodes.sort_values(["t", "node_id"]).reset_index(drop=True)\n\n\ndef load_candidates(folder: Path) -> tuple[pd.DataFrame, dict]:\n    """Read coordinates only, preserving one copy per frame across overlapping windows.\n\n    The earliest window containing a frame supplies its scores. Coordinates and IDs\n    must agree in other windows; window-dependent scores are counted in the receipt.\n    Features are intentionally never decompressed or loaded.\n    """\n    files = sorted(folder.glob("*.npz"))\n    if not files:\n        raise ValueError(f"No window NPZ files: {folder}")\n    frames: dict[int, pd.DataFrame] = {}\n    score_changes = 0\n    checkpoints = None\n    for path in files:\n        with np.load(path, allow_pickle=False) as saved:\n            meta = json.loads(saved["__metadata_json__"].tobytes().decode("utf-8"))\n            if meta.get("schema_version") != 1 or meta.get("dataset") != folder.name:\n                raise ValueError(f"Cache schema or dataset mismatch: {path}")\n            if meta.get("coordinate_units", {}).get("physical") != "micrometer":\n                raise ValueError(f"Expected physical coordinates in micrometers: {path}")\n            identity = tuple(\n                meta.get(f"{side}_checkpoint_sha256") for side in ["primary", "secondary"]\n            )\n            if checkpoints is not None and checkpoints != identity:\n                raise ValueError(f"Mixed model checkpoints: {path}")\n            checkpoints = identity\n            times = meta["window_frames"]\n            if len(times) != 2 or any(int(t) != t or t < 0 for t in times) or times[1] <= times[0]:\n                raise ValueError(f"Invalid window frames: {path}")\n            for side, t in zip(["src", "tgt"], times, strict=True):\n                ids = saved[f"candidate_ids_{side}"]\n                coords = saved[f"coords_{side}_physical"]\n                mask = saved[f"candidate_mask_{side}"]\n                scores = saved[f"detection_scores_{side}"]\n                n = len(ids)\n                if coords.shape != (n, 3) or mask.shape != (n,) or scores.shape != (n,):\n                    raise ValueError(f"Candidate array lengths disagree: {path}, {side}")\n                if mask.dtype != bool:\n                    raise ValueError(f"Candidate mask must be boolean: {path}")\n                frame = pd.DataFrame(coords[mask], columns=XYZ)\n                frame["node_id"] = ids[mask]\n                frame["t"] = int(t)\n                frame["score"] = scores[mask]\n                frame["window"] = path.stem\n                frame = validate_nodes(frame)\n                if not np.isfinite(frame.score).all() or not frame.score.between(0, 1).all():\n                    raise ValueError(f"Invalid detection score: {path}")\n                if t in frames:\n                    prior = frames[t]\n                    if not np.array_equal(prior[NODE_COLUMNS], frame[NODE_COLUMNS]):\n                        raise ValueError(\n                            f"Overlapping windows disagree on IDs/coordinates at t={t}"\n                        )\n                    score_changes += int(not np.array_equal(prior.score, frame.score))\n                else:\n                    frames[int(t)] = frame\n    result = validate_nodes(pd.concat(frames.values(), ignore_index=True))\n    receipt = {\n        "dataset": folder.name,\n        "windows": len(files),\n        "frames": sorted(frames),\n        "score_changed_frames": score_changes,\n        "checkpoints": checkpoints,\n        "score_policy": "earliest window containing each frame",\n        "units": "µm",\n    }\n    return result, receipt\n\n\ndef load_geff(path: Path, scale: tuple[float, float, float]) -> tuple[pd.DataFrame, pd.DataFrame]:\n    import zarr\n\n    if len(scale) != 3 or not np.isfinite(scale).all() or min(scale) <= 0:\n        raise ValueError("Voxel scale must have three positive finite values")\n    group = zarr.open_group(str(path), mode="r")\n    nodes = pd.DataFrame(\n        {\n            "node_id": np.asarray(group["nodes/ids"][:]),\n            **{axis: np.asarray(group[f"nodes/props/{axis}/values"][:]) for axis in ["t", *XYZ]},\n        }\n    )\n    nodes = validate_nodes(nodes)\n    nodes[XYZ] = nodes[XYZ].astype(float) * np.asarray(scale)\n    edges_array = np.asarray(group["edges/ids"][:])\n    if edges_array.ndim != 2 or edges_array.shape[1] != 2:\n        raise ValueError("GEFF edges/ids must have shape (N, 2)")\n    edges = pd.DataFrame(edges_array, columns=["source_id", "target_id"])\n    if edges.duplicated().any():\n        raise ValueError("Duplicate GEFF edges")\n    by_id = nodes.set_index("node_id")\n    if not set(edges.to_numpy().ravel()).issubset(by_id.index):\n        raise ValueError("GEFF edge references a missing node")\n    if len(edges):\n        ts = by_id.loc[edges.source_id, "t"].to_numpy()\n        tt = by_id.loc[edges.target_id, "t"].to_numpy()\n        if (tt <= ts).any():\n            raise ValueError("GEFF edges must point forward in time")\n    nodes["lineage"] = lineage_ids(nodes, edges)\n    nodes["division"] = nodes.node_id.isin(\n        edges.source_id.value_counts().loc[lambda s: s >= 2].index\n    )\n    return nodes, edges\n\n\ndef lineage_ids(nodes: pd.DataFrame, edges: pd.DataFrame) -> list[int]:\n    """Weak connected components retain both daughters at divisions."""\n    parent = {int(n): int(n) for n in nodes.node_id}\n\n    def root(n):\n        while parent[n] != n:\n            parent[n] = parent[parent[n]]\n            n = parent[n]\n        return n\n\n    for source, target in edges.itertuples(index=False, name=None):\n        a, b = root(int(source)), root(int(target))\n        parent[max(a, b)] = min(a, b)\n    return [root(int(n)) for n in nodes.node_id]\n\n\ndef match_frame(detections: pd.DataFrame, gt: pd.DataFrame, radius: float) -> pd.DataFrame:\n    """Maximum-cardinality one-to-one assignment, then minimum physical distance.\n\n    This is an exploratory diagnostic, not the official tracking metric. Matching\n    is performed before any display crop or lineage filter.\n    """\n    if not np.isfinite(radius) or radius <= 0:\n        raise ValueError("Matching radius must be positive and finite")\n    columns = [\n        "gt_id",\n        "detection_id",\n        "distance_um",\n        "nearest_distance_um",\n        "candidates_in_radius",\n    ]\n    if gt.empty:\n        return pd.DataFrame(columns=columns)\n    if len(set(gt.t) | set(detections.t)) > 1:\n        raise ValueError("match_frame accepts exactly one timepoint")\n    n, m = len(gt), len(detections)\n    rows = pd.DataFrame(\n        {\n            "gt_id": gt.node_id.to_numpy(),\n            "detection_id": pd.array([None] * n, dtype="Int64"),\n            "distance_um": np.nan,\n            "nearest_distance_um": np.nan,\n            "candidates_in_radius": np.zeros(n, dtype=int),\n        }\n    )\n    if not m:\n        return rows[columns]\n    distances = cdist(gt[XYZ].to_numpy(), detections[XYZ].to_numpy())\n    allowed = distances <= radius\n    # A forbidden pair costs more than all allowed distances combined, ensuring\n    # cardinality takes priority even where nearest-neighbor greedy matching fails.\n    penalty = (min(n, m) + 1) * (radius + 1)\n    gi, di = linear_sum_assignment(np.where(allowed, distances, penalty))\n    rows["nearest_distance_um"] = distances.min(axis=1)\n    rows["candidates_in_radius"] = allowed.sum(axis=1)\n    for g, d in zip(gi, di, strict=True):\n        if allowed[g, d]:\n            rows.loc[g, "detection_id"] = int(detections.iloc[d].node_id)\n            rows.loc[g, "distance_um"] = distances[g, d]\n    return rows[columns]\n\n\ndef analyze_sequence(detections, gt, frames, radius):\n    tables, counts = [], []\n    for t in frames:\n        det_t, gt_t = detections[detections.t == t], gt[gt.t == t]\n        matches = match_frame(det_t, gt_t, radius)\n        matches["t"] = t\n        tables.append(matches)\n        nmatch = int(matches.detection_id.notna().sum())\n        counts.append(\n            {\n                "t": t,\n                "検出候補": len(det_t),\n                "正解": len(gt_t),\n                "対応あり": nmatch,\n                "未対応の正解": len(gt_t) - nmatch,\n                "未対応の検出": len(det_t) - nmatch,\n                "正解の対応率": nmatch / len(gt_t) if len(gt_t) else np.nan,\n            }\n        )\n    return pd.concat(tables, ignore_index=True), pd.DataFrame(counts)\n\n\ndef image_array(path: Path):\n    import zarr\n\n    group = zarr.open(str(path), mode="r")\n    array = group if isinstance(group, zarr.Array) else group["0"]\n    if array.ndim != 4:\n        raise ValueError("Image must have shape (T, Z, Y, X)")\n    return array\n\n\ndef read_projection(path: Path, t: int, plane: str, z_range: tuple[int, int]):\n    """Load one timepoint only. Z bounds include both endpoint slices."""\n    array = image_array(path)\n    lo, hi = z_range\n    if not 0 <= t < array.shape[0] or not 0 <= lo <= hi < array.shape[1]:\n        raise ValueError("Requested timepoint or Z range is outside the image")\n    volume = np.asarray(array[t, lo : hi + 1, :, :])\n    axis = {"XY": 0, "XZ": 1, "YZ": 2}[plane]\n    return volume.max(axis=axis)\n', 'tracking_plots.py': '"""Plotly views shared by the local app and the Kaggle notebook."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nimport plotly.graph_objects as go\n\nfrom app.tracking_data import XYZ\n\nCOLORS = {\n    "gt": "#4ce3b0",\n    "miss": "#ff7889",\n    "det": "#76b7ff",\n    "other": "#8c97ab",\n    "link": "#f5cd73",\n    "trail": "#bc98ff",\n}\n\n\ndef spatial_figure(\n    detections,\n    gt,\n    edges,\n    matches,\n    *,\n    t,\n    plane="XY",\n    lineage=None,\n    trail=5,\n    z_bounds=None,\n    background=None,\n    scale=(1.625, 0.40625, 0.40625),\n    image_z_start=0,\n    show_unmatched=True,\n    show_links=True,\n    dataset="",\n):\n    det = detections[detections.t == t].copy()\n    now = gt[gt.t == t].copy()\n    pairs = matches[matches.t == t].dropna(subset=["detection_id"])\n    matched_gt, matched_det = set(pairs.gt_id), set(pairs.detection_id)\n    if z_bounds is not None:\n        det = det[det.z.between(*z_bounds)]\n        now = now[now.z.between(*z_bounds)]\n    if lineage is not None:\n        now = now[now.lineage == lineage]\n    is3d = plane == "3D"\n    horizontal, vertical = {"XY": ("x", "y"), "XZ": ("x", "z"), "YZ": ("y", "z"), "3D": ("x", "y")}[\n        plane\n    ]\n    figure = go.Figure()\n    if background is not None and not is3d:\n        high = float(np.percentile(background, 99.7))\n        axis_scale = dict(zip(XYZ, scale, strict=True))\n        figure.add_trace(\n            go.Heatmap(\n                z=background,\n                x=np.arange(background.shape[1]) * axis_scale[horizontal],\n                y=(np.arange(background.shape[0]) + (image_z_start if vertical == "z" else 0))\n                * axis_scale[vertical],\n                colorscale="Gray",\n                zmin=0,\n                zmax=max(high, 1),\n                showscale=False,\n                hoverinfo="skip",\n                name="画像",\n            )\n        )\n\n    def points(frame, name, color, symbol, size, kind):\n        if frame.empty:\n            return\n        custom = [\n            [kind, int(row.node_id), int(row.t), float(row.z), float(row.y), float(row.x)]\n            for row in frame.itertuples()\n        ]\n        kwargs = dict(\n            x=frame[horizontal],\n            y=frame[vertical],\n            mode="markers",\n            name=name,\n            marker=dict(size=size, color=color, symbol=symbol, line=dict(width=1)),\n            customdata=custom,\n            hovertemplate=(\n                "ID %{customdata[1]} · t=%{customdata[2]}<br>"\n                "z=%{customdata[3]:.2f} y=%{customdata[4]:.2f} x=%{customdata[5]:.2f} µm"\n                "<extra>%{fullData.name}</extra>"\n            ),\n        )\n        if is3d:\n            kwargs["z"] = frame.z\n            figure.add_trace(go.Scatter3d(**kwargs))\n        else:\n            figure.add_trace(go.Scatter(**kwargs))\n\n    def lines(segments, name, color, width, dash="solid"):\n        if not segments:\n            return\n        points_array = np.array([p for a, b in segments for p in [a, b, [np.nan] * 3]])\n        positions = dict(zip(XYZ, points_array.T, strict=True))\n        kwargs = dict(\n            x=positions[horizontal],\n            y=positions[vertical],\n            mode="lines",\n            name=name,\n            line=dict(color=color, width=width, dash=dash),\n            hoverinfo="skip",\n        )\n        if is3d:\n            figure.add_trace(go.Scatter3d(z=positions["z"], **kwargs))\n        else:\n            figure.add_trace(go.Scatter(**kwargs))\n\n    history = gt[gt.t.between(t - trail, t)]\n    if lineage is not None:\n        history = history[history.lineage == lineage]\n    if z_bounds is not None:\n        history = history[history.z.between(*z_bounds)]\n    by_id = history.set_index("node_id")\n    segments = [\n        (by_id.loc[s, XYZ].to_numpy(), by_id.loc[d, XYZ].to_numpy())\n        for s, d in edges.itertuples(index=False, name=None)\n        if s in by_id.index and d in by_id.index\n    ]\n    lines(segments, "正解の軌跡・分裂", COLORS["trail"], 3)\n    if show_links:\n        g_index, d_index = now.set_index("node_id"), det.set_index("node_id")\n        segments = [\n            (g_index.loc[r.gt_id, XYZ].to_numpy(), d_index.loc[int(r.detection_id), XYZ].to_numpy())\n            for r in pairs.itertuples()\n            if r.gt_id in g_index.index and int(r.detection_id) in d_index.index\n        ]\n        lines(segments, "同時刻の対応", COLORS["link"], 2, "dash")\n    if show_unmatched:\n        points(\n            det[~det.node_id.isin(matched_det)],\n            "未対応の検出",\n            COLORS["other"],\n            "circle-open",\n            5 if is3d else 8,\n            "detection",\n        )\n    points(\n        det[det.node_id.isin(matched_det)],\n        "対応ありの検出",\n        COLORS["det"],\n        "circle-open",\n        6 if is3d else 12,\n        "detection",\n    )\n    points(\n        now[now.node_id.isin(matched_gt)],\n        "対応ありの正解",\n        COLORS["gt"],\n        "circle",\n        4 if is3d else 7,\n        "gt",\n    )\n    points(\n        now[~now.node_id.isin(matched_gt)],\n        "未対応の正解",\n        COLORS["miss"],\n        "diamond",\n        5 if is3d else 10,\n        "gt",\n    )\n    if "division" in now:\n        points(now[now.division], "正解の分裂親", COLORS["trail"], "diamond-open", 12, "gt")\n    all_coords = pd.concat([detections[XYZ], gt[XYZ]])\n    ranges = {\n        axis: [\n            min(0.0, float(all_coords[axis].min()) - 2),\n            max(1.0, float(all_coords[axis].max()) + 2),\n        ]\n        for axis in XYZ\n    }\n    if background is not None and not is3d:\n        axis_scale = dict(zip(XYZ, scale, strict=True))\n        ranges[horizontal] = [\n            -axis_scale[horizontal] / 2,\n            (background.shape[1] - 0.5) * axis_scale[horizontal],\n        ]\n        offset = image_z_start if vertical == "z" else 0\n        ranges[vertical] = [\n            (offset - 0.5) * axis_scale[vertical],\n            (offset + background.shape[0] - 0.5) * axis_scale[vertical],\n        ]\n    figure.update_layout(\n        template="plotly_dark",\n        paper_bgcolor="#111827",\n        plot_bgcolor="#111827",\n        height=620,\n        title=f"{dataset} · t = {t} · {plane}",\n        margin=dict(l=40, r=20, t=60, b=65),\n        legend=dict(orientation="h", y=-0.12),\n        uirevision=f"{dataset}/{plane}",\n        xaxis=dict(title=f"{horizontal} (µm)", range=ranges[horizontal]),\n        yaxis=dict(\n            title=f"{vertical} (µm)", range=ranges[vertical][::-1], scaleanchor="x", scaleratio=1\n        ),\n        scene=dict(\n            xaxis=dict(title="x (µm)", range=ranges["x"]),\n            yaxis=dict(title="y (µm)", range=ranges["y"][::-1]),\n            zaxis=dict(title="z (µm)", range=ranges["z"]),\n            aspectmode="data",\n        ),\n    )\n    return figure\n\n\ndef timeline_figure(summary, t):\n    figure = go.Figure()\n    for col, color in [\n        ("正解", COLORS["gt"]),\n        ("対応あり", COLORS["det"]),\n        ("未対応の正解", COLORS["miss"]),\n    ]:\n        figure.add_trace(\n            go.Scatter(x=summary.t, y=summary[col], mode="lines", name=col, line=dict(color=color))\n        )\n    figure.add_vline(x=t, line_color=COLORS["link"])\n    figure.update_layout(\n        template="plotly_dark",\n        height=230,\n        margin=dict(l=40, r=15, t=15, b=30),\n        xaxis_title="時刻 t",\n        yaxis_title="細胞数",\n        legend=dict(orientation="h"),\n    )\n    return figure\n\n\ndef detail_table(gt, detections, matches, lineage=None):\n    table = gt.rename(columns={"node_id": "gt_id"}).merge(matches, on=["gt_id", "t"], how="left")\n    if lineage is not None:\n        table = table[table.lineage == lineage]\n    scores = (\n        detections.set_index("node_id").score if "score" in detections else pd.Series(dtype=float)\n    )\n    table["detection_score"] = table.detection_id.map(scores)\n    table["state"] = np.where(\n        ~table.t.isin(matches.t),\n        "キャッシュなし",\n        np.where(table.detection_id.notna(), "対応あり", "未対応"),\n    )\n    return table.sort_values(["t", "gt_id"])\n', 'tracking_notebook.py': '"""Interactive, CPU-only Kaggle/Jupyter entry point."""\n\nfrom __future__ import annotations\n\nimport html\nimport json\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport ipywidgets as widgets\nfrom IPython.display import HTML, display\n\nfrom app.tracking_data import (\n    analyze_sequence,\n    image_array,\n    load_candidates,\n    load_geff,\n    read_projection,\n)\nfrom app.tracking_plots import detail_table, spatial_figure, timeline_figure\n\n\ndef find_input_directory(root: Path, name: str) -> Path:\n    matches = []\n    for depth in range(5):\n        matches.extend(p for p in root.glob("*/" * depth + name) if p.is_dir())\n    unique = sorted(set(matches))\n    if len(unique) != 1:\n        raise ValueError(f"Expected one {name} directory; found {unique}. Set the path explicitly.")\n    return unique[0]\n\n\ndef plotly_iframe(figure, height: int) -> str:\n    """Render Plotly in an iframe that also works inside a Kaggle HTML widget."""\n    document = figure.to_html(\n        full_html=True,\n        include_plotlyjs="cdn",\n        config={"responsive": True, "displaylogo": False},\n    )\n    return (\n        f\'<iframe title="Plotly figure" srcdoc="{html.escape(document, quote=True)}" \'\n        f\'style="width:100%;height:{height}px;border:0" loading="eager"></iframe>\'\n    )\n\n\nclass TrackingNotebookViewer:\n    def __init__(self, cache_root: Path, train_root: Path, config: dict):\n        self.cache_root, self.train_root, self.config = cache_root, train_root, config\n        datasets = sorted(\n            p.name\n            for p in cache_root.iterdir()\n            if p.is_dir() and any(p.glob("*.npz")) and (train_root / f"{p.name}.geff").is_dir()\n        )\n        if not datasets:\n            raise ValueError("No matching cache/GEFF pairs. Attach competition and exp015 outputs.")\n        style = {"description_width": "initial"}\n        train_count = sum(path.is_dir() for path in train_root.glob("*.geff"))\n        self.dataset = widgets.Dropdown(\n            options=datasets, description=f"サンプル ({len(datasets)}件)", style=style\n        )\n        self.dataset_note = widgets.HTML(\n            value=(\n                f"<small>検出キャッシュあり: {len(datasets)}件 / "\n                f"正解 train: {train_count}件。"\n                "このビューアーでは検出キャッシュがあるサンプルだけを選べます。</small>"\n            )\n        )\n        self.frame = widgets.SelectionSlider(\n            options=[0], description="時刻 t", continuous_update=False\n        )\n        self.play = widgets.Play(\n            min=0,\n            max=0,\n            interval=int(config["playback_interval_seconds"] * 1000),\n            layout=widgets.Layout(display="none"),\n        )\n        self.previous_frame = widgets.Button(description="前の時刻")\n        self.play_pause = widgets.Button(description="再生")\n        self.next_frame = widgets.Button(description="次の時刻")\n        self.plane = widgets.Dropdown(options=["XY", "XZ", "YZ", "3D"], description="表示")\n        self.lineage = widgets.Dropdown(\n            options=[("すべて", -1)], description="正解の系譜", style=style\n        )\n        self.node = widgets.Dropdown(options=[("選択なし", -1)], description="正解 ID", style=style)\n        self.zrange = widgets.IntRangeSlider(\n            value=[0, 63], min=0, max=63, description="Z断面", continuous_update=False\n        )\n        self.radius = widgets.BoundedFloatText(\n            value=config["matching_radius_um"],\n            min=0.01,\n            max=1000.0,\n            description="対応距離 µm",\n            style=style,\n        )\n        self.score = widgets.FloatSlider(\n            value=config["minimum_detection_score"],\n            min=0,\n            max=1,\n            step=0.01,\n            description="検出score ≥",\n            style=style,\n            continuous_update=False,\n        )\n        self.trail = widgets.IntSlider(\n            value=config["trail_frames"],\n            min=0,\n            max=30,\n            description="軌跡の過去frame数",\n            style=style,\n            continuous_update=False,\n        )\n        self.background = widgets.Checkbox(value=True, description="画像を重ねる")\n        self.unmatched = widgets.Checkbox(value=True, description="未対応の検出を表示")\n        self.links = widgets.Checkbox(value=True, description="対応線を表示")\n        self.prev_issue = widgets.Button(description="前の未対応時刻")\n        self.next_issue = widgets.Button(description="次の未対応時刻")\n        self.export = widgets.Button(description="対応表を CSV 保存")\n        self.message = widgets.HTML()\n        full_width = widgets.Layout(width="100%")\n        self.summary_output = widgets.HTML(layout=full_width)\n        self.spatial_output = widgets.HTML(layout=full_width)\n        self.timeline_output = widgets.HTML(layout=full_width)\n        self.detail_output = widgets.HTML(layout=full_width)\n        self.busy = False\n        self._projection = lru_cache(maxsize=6)(read_projection)\n        self.dataset.observe(self._load, "value")\n        for w in [self.radius, self.score]:\n            w.observe(self._analyze, "value")\n        for w in [\n            self.frame,\n            self.plane,\n            self.lineage,\n            self.zrange,\n            self.trail,\n            self.background,\n            self.unmatched,\n            self.links,\n        ]:\n            w.observe(self._render, "value")\n        self.node.observe(self._select_node, "value")\n        self.play.observe(self._play_frame, "value")\n        self.play.observe(self._sync_play_button, "playing")\n        self.previous_frame.on_click(lambda _: self._step_frame(-1))\n        self.play_pause.on_click(self._toggle_play)\n        self.next_frame.on_click(lambda _: self._step_frame(1))\n        self.prev_issue.on_click(lambda _: self._jump(-1))\n        self.next_issue.on_click(lambda _: self._jump(1))\n        self.export.on_click(self._export)\n        self._load()\n\n    def _sync_play_button(self, change):\n        self.play_pause.description = "一時停止" if change["new"] else "再生"\n\n    def _toggle_play(self, _=None):\n        self.play.playing = not self.play.playing\n\n    def _step_frame(self, direction):\n        self.play.playing = False\n        index = self.frames.index(self.frame.value)\n        target = min(max(index + direction, 0), len(self.frames) - 1)\n        self.frame.value = self.frames[target]\n\n    def _load(self, _=None):\n        self.busy = True\n        try:\n            self.play.playing = False\n            name = self.dataset.value\n            self.detections, self.receipt = load_candidates(self.cache_root / name)\n            self.gt, self.edges = load_geff(\n                self.train_root / f"{name}.geff", tuple(self.config["voxel_scale_um"])\n            )\n            self.frames = self.receipt["frames"]\n            self.frame.options = self.frames\n            self.frame.value = self.frames[0]\n            self.play.value, self.play.max = 0, len(self.frames) - 1\n            self.image_path = self.train_root / f"{name}.zarr"\n            self.shape = image_array(self.image_path).shape if self.image_path.exists() else None\n            self.background.disabled = self.shape is None\n            self.background.value = self.shape is not None\n            depth = (\n                self.shape[1]\n                if self.shape\n                else max(1, int(self.gt.z.max() / self.config["voxel_scale_um"][0]) + 1)\n            )\n            self.zrange.value = (0, 0)\n            self.zrange.max = depth - 1\n            self.zrange.value = (0, depth - 1)\n            self.lineage.options = [("すべて", -1)] + [\n                (str(n), int(n)) for n in sorted(self.gt.lineage.unique())\n            ]\n            self.lineage.value = -1\n            self.node.options = [("選択なし", -1)] + [\n                (f"{r.node_id} · t={r.t}" + (" · 分裂" if r.division else ""), int(r.node_id))\n                for r in self.gt.itertuples()\n            ]\n            self.node.value = -1\n            self._projection.cache_clear()\n        finally:\n            self.busy = False\n        self._analyze()\n\n    def _analyze(self, _=None):\n        if self.busy:\n            return\n        self.filtered = self.detections[self.detections.score >= self.score.value]\n        self.matches, self.summary = analyze_sequence(\n            self.filtered, self.gt, self.frames, self.radius.value\n        )\n        self._render()\n\n    def _play_frame(self, change):\n        if not self.busy:\n            self.frame.value = self.frames[change["new"]]\n\n    def _select_node(self, _=None):\n        if self.busy or self.node.value == -1:\n            return\n        row = self.gt[self.gt.node_id == self.node.value].iloc[0]\n        self.busy = True\n        self.lineage.value = int(row.lineage)\n        if int(row.t) in self.frames:\n            self.frame.value = int(row.t)\n        self.busy = False\n        self._render()\n\n    def _jump(self, direction):\n        missing = self.matches[self.matches.detection_id.isna()]\n        if self.lineage.value != -1:\n            missing = missing[\n                missing.gt_id.isin(self.gt[self.gt.lineage == self.lineage.value].node_id)\n            ]\n        times = sorted(set(missing.t))\n        eligible = [t for t in times if (t - self.frame.value) * direction > 0]\n        if eligible:\n            self.frame.value = eligible[0] if direction > 0 else eligible[-1]\n\n    def _export(self, _=None):\n        path = Path.cwd() / f"tracking_matches_{self.dataset.value}.csv"\n        detail_table(self.gt, self.filtered, self.matches).to_csv(path, index=False)\n        receipt = {\n            **self.receipt,\n            "radius_um": self.radius.value,\n            "minimum_score": self.score.value,\n            "voxel_scale_um": self.config["voxel_scale_um"],\n            "diagnostic_only": True,\n        }\n        path.with_suffix(".json").write_text(json.dumps(receipt, ensure_ascii=False, indent=2))\n        self.message.value = f"保存: {html.escape(str(path))}（同名 JSON に設定を保存）"\n\n    def _render(self, _=None):\n        if self.busy:\n            return\n        # Keep resume playback at the time selected by the slider or a GT node.\n        self.busy = True\n        self.play.value = self.frames.index(self.frame.value)\n        self.busy = False\n        try:\n            t, plane = self.frame.value, self.plane.value\n            scale = tuple(self.config["voxel_scale_um"])\n            lo, hi = self.zrange.value\n            bounds = ((lo - 0.5) * scale[0], (hi + 0.5) * scale[0])\n            background = None\n            if self.background.value and plane != "3D" and self.shape and t < self.shape[0]:\n                background = self._projection(self.image_path, t, plane, (lo, hi))\n            row = self.summary[self.summary.t == t].iloc[0]\n            self.summary_output.value = (\n                f"<b>t={t}</b>\u3000検出 {int(row[\'検出候補\'])}\u3000正解 {int(row[\'正解\'])}\u3000"\n                f"対応 {int(row[\'対応あり\'])}\u3000未対応の正解 {int(row[\'未対応の正解\'])}"\n                "<br><small>件数は全視野・全Z。表示範囲や系譜の選択は対応計算を変えません。</small>"\n            )\n            lineage = None if self.lineage.value == -1 else self.lineage.value\n            self.figure = spatial_figure(\n                self.filtered,\n                self.gt,\n                self.edges,\n                self.matches,\n                t=t,\n                plane=plane,\n                lineage=lineage,\n                trail=self.trail.value,\n                z_bounds=bounds,\n                background=background,\n                scale=scale,\n                image_z_start=lo,\n                show_unmatched=self.unmatched.value,\n                show_links=self.links.value,\n                dataset=self.dataset.value,\n            )\n            self.spatial_output.value = plotly_iframe(self.figure, 640)\n            self.timeline_output.value = plotly_iframe(timeline_figure(self.summary, t), 250)\n            details = detail_table(self.gt, self.filtered, self.matches, lineage)\n            columns = [\n                "t",\n                "gt_id",\n                "detection_id",\n                "distance_um",\n                "nearest_distance_um",\n                "candidates_in_radius",\n                "detection_score",\n                "division",\n                "state",\n            ]\n            table = details[columns] if lineage is not None else details[details.t == t][columns]\n            self.detail_output.value = (\n                \'<div style="max-height:420px;overflow:auto">\'\n                + table.to_html(index=False, border=0)\n                + "</div>"\n            )\n        except (ValueError, KeyError, OSError, IndexError) as exc:\n            self.summary_output.value = f"<b>読み込み・表示エラー:</b> {html.escape(str(exc))}"\n            self.spatial_output.value = ""\n            self.timeline_output.value = ""\n            self.detail_output.value = ""\n            raise\n\n    def show(self):\n        display(\n            HTML(\n                "<h2>Cell tracking · 検出と正解の時間変化</h2>"\n                "<p>正解は疎です。未対応の検出は誤検出を意味しません。"\n                "対応は同時刻・3次元の距離による1対1割当で、公式スコアではありません。"\n                "候補IDは各検出点のIDで、細胞の軌跡IDではありません。</p>"\n            )\n        )\n        display(\n            widgets.VBox(\n                [\n                    widgets.HBox([self.dataset, self.plane]),\n                    self.dataset_note,\n                    self.play,\n                    widgets.HBox(\n                        [self.previous_frame, self.play_pause, self.next_frame, self.frame]\n                    ),\n                    widgets.HBox([self.prev_issue, self.next_issue]),\n                    widgets.HBox([self.lineage, self.node]),\n                    widgets.HBox([self.zrange, self.trail]),\n                    widgets.HBox([self.radius, self.score]),\n                    widgets.HBox([self.background, self.unmatched, self.links, self.export]),\n                    self.message,\n                    self.summary_output,\n                    self.spatial_output,\n                    self.timeline_output,\n                    self.detail_output,\n                ]\n            )\n        )\n        return self\n', 'tracking_viewer.yaml': '# Local EDA settings; these do not alter experiment configuration.\nvoxel_scale_um: [1.625, 0.40625, 0.40625]\nmatching_radius_um: 7.0\nminimum_detection_score: 0.0\ntrail_frames: 5\nplayback_interval_seconds: 0.7\n'}
for filename, source in FILES.items():
    (PACKAGE / 'app' / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, str(PACKAGE))
import subprocess
# Existing support input supplies Zarr wheels; never access the package index.
INPUT_ROOT = Path("/kaggle/input")
wheel_dirs = []
for depth in range(6):
    wheel_dirs.extend(p for p in INPUT_ROOT.glob("*/" * depth + "wheels") if p.is_dir())
if not wheel_dirs:
    raise FileNotFoundError("Attach pilkwang/biohub-tracking-support-pack-50ep-v1")
command = [sys.executable, "-m", "pip", "install", "--no-index", "zarr>=3.0.10,<4"]
for directory in sorted(wheel_dirs):
    command.extend(["--find-links", str(directory)])
subprocess.run(command, check=True)
import numpy, pandas, scipy, plotly, ipywidgets, zarr
print({m.__name__: m.__version__ for m in [numpy, pandas, scipy, plotly, ipywidgets, zarr]})


In [ ]:
import yaml
from app.tracking_notebook import TrackingNotebookViewer, find_input_directory

# 自動検出が曖昧な場合だけ、下の2行を明示パスへ変更してください。
INPUT = Path('/kaggle/input')
CACHE_ROOT = find_input_directory(INPUT, 'window_cache')
TRAIN_ROOT = find_input_directory(INPUT, 'train')
CONFIG = yaml.safe_load((PACKAGE / 'app' / 'tracking_viewer.yaml').read_text())
print('cache:', CACHE_ROOT)
print('train:', TRAIN_ROOT)
print('settings:', CONFIG)


In [ ]:
import plotly.io as pio
# 保存ページ末尾の静的Figure用。対話図はHTML widget内のiframeで描画します。
pio.renderers.default = 'kaggle'
import faulthandler
faulthandler.dump_traceback_later(90)
try:
    print("Loading first image and viewer...", flush=True)
    viewer = TrackingNotebookViewer(CACHE_ROOT, TRAIN_ROOT, CONFIG)
    viewer.show()
    print("Initial viewer ready", flush=True)
finally:
    faulthandler.cancel_dump_traceback_later()


In [ ]:
# batch 保存時も入力・対応計算・初期表示の到達点を確認できます。
import json
from app.tracking_plots import detail_table
receipt = {**viewer.receipt, 'gt_nodes': len(viewer.gt), 'gt_edges': len(viewer.edges),
           'matched_gt': int(viewer.matches.detection_id.notna().sum()),
           'image_shape': list(viewer.shape) if viewer.shape else None,
           'matching_radius_um': viewer.radius.value, 'diagnostic_only': True}
Path('viewer_input_check.json').write_text(json.dumps(receipt, indent=2))
initial_matches = detail_table(viewer.gt, viewer.filtered, viewer.matches)
initial_matches.to_csv('viewer_initial_matches.csv', index=False)
print(receipt)
# 静的ページにも初期フレームを残します。操作時は Edit → Run All を使ってください。
display(viewer.figure)
